## Author: Gabriel Vance
## Class: DSC540 - Data Preparation
## Professor: Catherine Williams
## Date: July 27, 2025

# Purpose: This notebook performs various tasks for Milestone 3 of the term project.
# This week focuses on Cleaning/Formatting API Data

In [44]:
import requests
import json

api_key = "fH8OUy9MPiGaQRKQzwTlQ4xEgYvgfv6qPbK4RE9Z"

# update the URL to include the API key as a query parameter
url = f"https://api.eia.gov/v2/seds/data/?api_key={api_key}&frequency=annual&data[0]=value&facets[seriesId][]=CLTCB&facets[seriesId][]=ELGBP&facets[seriesId][]=MGTCP&facets[seriesId][]=NGSCB&facets[seriesId][]=NUETB&facets[seriesId][]=TEACE&start=1996&end=2013&sort[0][column]=period&sort[0][direction]=desc&offset=0&length=5000"

# redefine x_params and headers if needed
x_params = {
    "frequency": "annual",
    "data": ["value"],
    "facets": {
        "seriesId": [
            "CLTCB", 
            "ELGBP",
            "MGTCP",
            "NGSCB", 
            "NUETB",
            "TEACE" #  Total energy CO2 emissions for the transportation sector
        ]
    },
    "start": "1996",
    "end": "2013",
    "sort": [
        {
            "column": "period",
            "direction": "desc"
        }
    ],
    "offset": 0,
    "length": 5000
}

headers = {"X-Params": json.dumps(x_params)}

response = requests.get(url, headers=headers)
print(response.text)

{"response":{"total":"4732","dateFormat":"YYYY","frequency":"annual","data":[{"period":"2013","seriesId":"TEACE","seriesDescription":"Total energy CO2 emissions for the transportation sector","stateId":"OK","stateDescription":"Oklahoma","value":"29.9","unit":"Million metric tons CO2"},{"period":"2013","seriesId":"TEACE","seriesDescription":"Total energy CO2 emissions for the transportation sector","stateId":"IA","stateDescription":"Iowa","value":"19.9","unit":"Million metric tons CO2"},{"period":"2013","seriesId":"TEACE","seriesDescription":"Total energy CO2 emissions for the transportation sector","stateId":"MI","stateDescription":"Michigan","value":"50.3","unit":"Million metric tons CO2"},{"period":"2013","seriesId":"TEACE","seriesDescription":"Total energy CO2 emissions for the transportation sector","stateId":"OR","stateDescription":"Oregon","value":"20.4","unit":"Million metric tons CO2"},{"period":"2013","seriesId":"TEACE","seriesDescription":"Total energy CO2 emissions for the t

## Transformation 1 - Convert Response to DataFrame



In [45]:
import pandas as pd

# parse the JSON content from the API response
json_data = response.json()

data = json_data['response']['data']
df = pd.DataFrame(data)

df.head(5)

,period,seriesId,seriesDescription,stateId,stateDescription,value,unit
0,2013,TEACE,Total energy CO2 emissions for the transportat...,OK,Oklahoma,29.9,Million metric tons CO2
1,2013,TEACE,Total energy CO2 emissions for the transportat...,IA,Iowa,19.9,Million metric tons CO2
2,2013,TEACE,Total energy CO2 emissions for the transportat...,MI,Michigan,50.3,Million metric tons CO2
3,2013,TEACE,Total energy CO2 emissions for the transportat...,OR,Oregon,20.4,Million metric tons CO2
4,2013,TEACE,Total energy CO2 emissions for the transportat...,KS,Kansas,19,Million metric tons CO2


## Transformation 2 - Drop unnecessary columns, rename columns

In [46]:
# We don't need ID as it's the same as series description. stateId is the 2 character state code.
df = df.drop(columns=['seriesId','stateId'])

In [47]:
df.rename(columns={"period": "Year", "stateDescription": "State"}, inplace=True)
df.head()

,Year,seriesDescription,State,value,unit
0,2013,Total energy CO2 emissions for the transportat...,Oklahoma,29.9,Million metric tons CO2
1,2013,Total energy CO2 emissions for the transportat...,Iowa,19.9,Million metric tons CO2
2,2013,Total energy CO2 emissions for the transportat...,Michigan,50.3,Million metric tons CO2
3,2013,Total energy CO2 emissions for the transportat...,Oregon,20.4,Million metric tons CO2
4,2013,Total energy CO2 emissions for the transportat...,Kansas,19,Million metric tons CO2


In [48]:
# Print unique values for series descriptions and units
unique_series = df['seriesDescription'].unique()
unique_units = df['unit'].unique()

print("Unique series:")
for series in unique_series:
    print(series)

print("\nUnique units:")
for unit in unique_units:
    print(unit)

Unique series:
Total energy CO2 emissions for the transportation sector
Coal total consumption
Total (all fuels) electric generating units net summer capacity in all sectors
Natural gas total consumption adjusted for process fuel
Nuclear energy consumed for electricity generation, total
Motor gasoline total consumption

Unique units:
Million metric tons CO2
Billion Btu
Thousand kilowatts
Thousand barrels


## Transformation 3 - Combine Series Description and Units

In [49]:
df['Energy Consumption'] = df['seriesDescription'] + " (" + df['unit'] + ")"
df.head()

,Year,seriesDescription,State,value,unit,Energy Consumption
0,2013,Total energy CO2 emissions for the transportat...,Oklahoma,29.9,Million metric tons CO2,Total energy CO2 emissions for the transportat...
1,2013,Total energy CO2 emissions for the transportat...,Iowa,19.9,Million metric tons CO2,Total energy CO2 emissions for the transportat...
2,2013,Total energy CO2 emissions for the transportat...,Michigan,50.3,Million metric tons CO2,Total energy CO2 emissions for the transportat...
3,2013,Total energy CO2 emissions for the transportat...,Oregon,20.4,Million metric tons CO2,Total energy CO2 emissions for the transportat...
4,2013,Total energy CO2 emissions for the transportat...,Kansas,19,Million metric tons CO2,Total energy CO2 emissions for the transportat...


In [50]:
# Print unique values for Energy Consumption
unique_series = df['Energy Consumption'].unique()

print("Unique Energy Consumption values:")
for series in unique_series:
    print(series)



Unique Energy Consumption values:
Total energy CO2 emissions for the transportation sector (Million metric tons CO2)
Coal total consumption (Billion Btu)
Total (all fuels) electric generating units net summer capacity in all sectors (Thousand kilowatts)
Natural gas total consumption adjusted for process fuel (Billion Btu)
Nuclear energy consumed for electricity generation, total (Billion Btu)
Motor gasoline total consumption (Thousand barrels)


## Transformation 4 - Create new columns based on unique values in Series Description

In [51]:
# Pivot the DataFrame so that each unique value in "Energy Consumption" becomes its own column
df_wide = df.pivot_table(index=['Year', 'State'], 
                         columns='Energy Consumption', 
                         values='value', 
                         aggfunc='first').reset_index()

# Optionally, convert the new columns to numeric values
cols = [col for col in df_wide.columns if col not in ['Year','State']]
df_wide[cols] = df_wide[cols].apply(pd.to_numeric, errors='coerce')

df_wide

Energy Consumption,Year,State,Coal total consumption (Billion Btu),Motor gasoline total consumption (Thousand barrels),Natural gas total consumption adjusted for process fuel (Billion Btu),"Nuclear energy consumed for electricity generation, total (Billion Btu)",Total (all fuels) electric generating units net summer capacity in all sectors (Thousand kilowatts),Total energy CO2 emissions for the transportation sector (Million metric tons CO2)
0,1997,Alabama,867264,55694,278364,310336,NaN,31.1
1,1997,Alaska,11719,6312,120878,0,NaN,13.5
2,1997,Arizona,369872,48884,118515,307623,NaN,27.7
3,1997,Arkansas,246786,33184,225913,149100,NaN,19.3
4,1997,California,82651,322871,2020325,320194,NaN,199.2
...,...,...,...,...,...,...,...,...
879,2013,Virginia,290508,92808,416068,306430,24828.0,50.2
880,2013,Washington,74955,65300,282870,88408,30656.0,40.4
881,2013,West Virginia,771188,18791,85153,0,16282.0,11.0
882,2013,Wisconsin,454554,58846,449724,121994,17342.0,28.0


## Transformation 5 - Handle NaN/Missing values

In [52]:
# Identify the numeric columns to fill NaN values (exclude non-numeric columns)
numeric_cols = [col for col in df_wide.columns if col not in ['Year', 'State']]

# Group by 'State' and fill NaN values with the mean of each group
df_wide[numeric_cols] = df_wide.groupby('State')[numeric_cols].transform(lambda x: x.fillna(x.mean()))

# Display the updated DataFrame
df_wide

Energy Consumption,Year,State,Coal total consumption (Billion Btu),Motor gasoline total consumption (Thousand barrels),Natural gas total consumption adjusted for process fuel (Billion Btu),"Nuclear energy consumed for electricity generation, total (Billion Btu)",Total (all fuels) electric generating units net summer capacity in all sectors (Thousand kilowatts),Total energy CO2 emissions for the transportation sector (Million metric tons CO2)
0,1997,Alabama,867264,55694,278364,310336,32084.166667,31.1
1,1997,Alaska,11719,6312,120878,0,2107.333333,13.5
2,1997,Arizona,369872,48884,118515,307623,26854.666667,27.7
3,1997,Arkansas,246786,33184,225913,149100,15603.333333,19.3
4,1997,California,82651,322871,2020325,320194,68462.833333,199.2
...,...,...,...,...,...,...,...,...
879,2013,Virginia,290508,92808,416068,306430,24828.000000,50.2
880,2013,Washington,74955,65300,282870,88408,30656.000000,40.4
881,2013,West Virginia,771188,18791,85153,0,16282.000000,11.0
882,2013,Wisconsin,454554,58846,449724,121994,17342.000000,28.0


In [53]:
df_wide[numeric_cols] = df_wide[numeric_cols].round(2)
df_wide

Energy Consumption,Year,State,Coal total consumption (Billion Btu),Motor gasoline total consumption (Thousand barrels),Natural gas total consumption adjusted for process fuel (Billion Btu),"Nuclear energy consumed for electricity generation, total (Billion Btu)",Total (all fuels) electric generating units net summer capacity in all sectors (Thousand kilowatts),Total energy CO2 emissions for the transportation sector (Million metric tons CO2)
0,1997,Alabama,867264,55694,278364,310336,32084.17,31.1
1,1997,Alaska,11719,6312,120878,0,2107.33,13.5
2,1997,Arizona,369872,48884,118515,307623,26854.67,27.7
3,1997,Arkansas,246786,33184,225913,149100,15603.33,19.3
4,1997,California,82651,322871,2020325,320194,68462.83,199.2
...,...,...,...,...,...,...,...,...
879,2013,Virginia,290508,92808,416068,306430,24828.00,50.2
880,2013,Washington,74955,65300,282870,88408,30656.00,40.4
881,2013,West Virginia,771188,18791,85153,0,16282.00,11.0
882,2013,Wisconsin,454554,58846,449724,121994,17342.00,28.0


## 1 paragraph of the ethical implications of data wrangling specific to your datasource and the steps you completed answering the following questions:
